<!--
Project: Heart Disease Prediction
File: heart_disease_prediction.ipynb
Description: Binary classification of heart disease using Decision Tree and Random Forest.
Course: 4IZ210 Zpracování informací a znalostí (strojové učení I)
Authors: Jan Alexandr Kopřiva <jan.alexandr.kopriva@gmail.com>,
         David Hložek, Jakub Hermann, Ondřej Čech, Milan Tvrdík
License: MIT
-->

# Heart Disease Prediction

Binary classification of heart disease presence using Decision Tree and Random Forest.

## Contents
1. Setup
2. Data Loading
3. Exploratory Data Analysis
4. Preprocessing
5. Model Training
6. Evaluation
7. Interpretation


## 1. Setup


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import set_config

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

os.makedirs("outputs/images", exist_ok=True)
os.makedirs("outputs/data", exist_ok=True)


## 2. Data Loading


In [ ]:
heart_data = pd.read_csv('heart.csv')
print(f"Shape: {heart_data.shape}")
heart_data.head()


In [ ]:
heart_data.info()


In [ ]:
heart_data.describe()


### Cost Matrix

Missing a disease (FN) is 10x worse than a false alarm (FP) in clinical context.


In [ ]:
# Asymmetric costs: FN=10 (missed diagnosis), FP=1 (unnecessary follow-up)
cost_matrix = np.array([[0, 1], [10, 0]])  # [[TN, FP], [FN, TP]]
print("Cost Matrix:\n", cost_matrix)


## 3. Exploratory Data Analysis

### 3.1 Age Distribution by Heart Disease


In [ ]:
bins = list(range(0, 105, 5))
labels = [f"{i}-{i+4}" for i in range(0, 100, 5)]

age_groups = heart_data.groupby(
    [pd.cut(heart_data['Age'], bins=bins, labels=labels, right=False), "HeartDisease"],
    observed=True
).size().unstack("HeartDisease")
age_groups.columns = age_groups.columns.map({0: "No", 1: "Yes"})

fig, ax = plt.subplots(figsize=(12, 6))
age_groups.plot.bar(stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title("Age Distribution by Heart Disease", fontsize=14)
ax.set_xlabel('Age Group')
ax.set_ylabel('Count')
ax.legend(title='Heart Disease')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/images/age_distribution_heart_disease.jpg", dpi=150)
plt.show()


### 3.2 Feature Distributions


In [ ]:
cols_of_interest = ['MaxHR', 'RestingBP', 'Cholesterol']

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))
for col, ax in zip(cols_of_interest, axes.flatten()):
    heart_data[col].hist(ax=ax, bins=50, color='#3498db', edgecolor='white')
    ax.set_title(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig("outputs/images/columns_histograms.jpg", dpi=150)
plt.show()


### 3.3 Correlation Matrix


In [ ]:
heart_data_num = heart_data.select_dtypes(['number'])

fig = plt.figure(figsize=(10, 8))
plt.matshow(heart_data_num.corr(), fignum=fig.number, cmap='RdBu_r')
plt.xticks(range(heart_data_num.shape[1]), heart_data_num.columns, fontsize=11, rotation=45, ha='left')
plt.yticks(range(heart_data_num.shape[1]), heart_data_num.columns, fontsize=11)
plt.colorbar(fraction=0.046, pad=0.04)
plt.title('Correlation Matrix', fontsize=14, pad=20)
plt.savefig("outputs/images/correlation_matrix.jpg", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Absolute correlation with target
correlation = pd.DataFrame(
    heart_data_num.corr()["HeartDisease"]
    .abs()
    .drop(["HeartDisease"])
    .sort_values(ascending=False)
)
correlation.columns = ['Abs Correlation']
correlation


## 4. Preprocessing

### 4.1 Train-Test Split


In [ ]:
X = heart_data.drop('HeartDisease', axis=1)
y = heart_data['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")


### 4.2 Feature Transformation

StandardScaler for numeric, OneHotEncoder for categorical.


In [ ]:
set_config(transform_output="pandas")  # Keep feature names after transform

numeric_features = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
categorical_features = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
    ],
    verbose_feature_names_out=False
)

preprocessor.fit(X_train)
X_train_preprocessed = preprocessor.transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print(f"Features after preprocessing: {X_train_preprocessed.shape[1]}")


In [ ]:
X_train_preprocessed.head()


## 5. Model Training

### 5.1 Decision Tree


In [ ]:
param_grid_dt = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search_dt.fit(X_train_preprocessed, y_train)

print(f"Best params: {grid_search_dt.best_params_}")
print(f"CV score: {grid_search_dt.best_score_:.4f}")


In [ ]:
# Using default params here; grid search above is for reference
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_preprocessed, y_train)
dt_predictions = dt_model.predict(X_test_preprocessed)


### 5.2 Random Forest


In [ ]:
param_grid_rf = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search_rf.fit(X_train_preprocessed, y_train)

print(f"Best params: {grid_search_rf.best_params_}")
print(f"CV score: {grid_search_rf.best_score_:.4f}")


In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_preprocessed, y_train)
rf_predictions = rf_model.predict(X_test_preprocessed)


### 5.3 Baseline (Dummy Classifier)


In [ ]:
# Random predictions as baseline to verify models add value
dummy_model = DummyClassifier(strategy="uniform")
dummy_model.fit(X_train_preprocessed, y_train)
dummy_predictions = dummy_model.predict(X_test_preprocessed)


## 6. Evaluation

### 6.1 Decision Tree


In [ ]:
print(classification_report(y_test, dt_predictions))
print(f"Accuracy: {accuracy_score(y_test, dt_predictions):.4f}")

dt_conf = confusion_matrix(y_test, dt_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(dt_conf, display_labels=['Healthy', 'Disease']).plot(ax=ax, cmap='Blues')
ax.set_title('Decision Tree')
plt.savefig("outputs/images/dt_confusion_matrix.jpg", dpi=150)
plt.show()

dt_cost = (dt_conf * cost_matrix).sum()
print(f"Total cost: {dt_cost}")


### 6.2 Random Forest


In [ ]:
print(classification_report(y_test, rf_predictions))
print(f"Accuracy: {accuracy_score(y_test, rf_predictions):.4f}")

rf_conf = confusion_matrix(y_test, rf_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(rf_conf, display_labels=['Healthy', 'Disease']).plot(ax=ax, cmap='Greens')
ax.set_title('Random Forest')
plt.savefig("outputs/images/rf_confusion_matrix.jpg", dpi=150)
plt.show()

rf_cost = (rf_conf * cost_matrix).sum()
print(f"Total cost: {rf_cost}")


### 6.3 Baseline


In [ ]:
print(classification_report(y_test, dummy_predictions))
print(f"Accuracy: {accuracy_score(y_test, dummy_predictions):.4f}")

dummy_conf = confusion_matrix(y_test, dummy_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(dummy_conf, display_labels=['Healthy', 'Disease']).plot(ax=ax, cmap='Oranges')
ax.set_title('Dummy Classifier')
plt.savefig("outputs/images/dummy_confusion_matrix.jpg", dpi=150)
plt.show()

dummy_cost = (dummy_conf * cost_matrix).sum()
print(f"Total cost: {dummy_cost}")


### 6.4 Model Comparison


In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Dummy'],
    'Accuracy': [
        accuracy_score(y_test, dt_predictions),
        accuracy_score(y_test, rf_predictions),
        accuracy_score(y_test, dummy_predictions)
    ],
    'Cost': [dt_cost, rf_cost, dummy_cost]
}).sort_values('Accuracy', ascending=False)

results


## 7. Interpretation

### 7.1 Decision Tree Visualization


In [ ]:
# Truncated to depth=3 for readability
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_model, max_depth=3, feature_names=X_train_preprocessed.columns, 
          class_names=['Healthy', 'Disease'], filled=True, rounded=True, ax=ax)
plt.savefig("outputs/images/decision_tree_plot.jpg", dpi=150, bbox_inches='tight')
plt.show()


### 7.2 Feature Importance


In [ ]:
dt_importances = pd.Series(
    dt_model.feature_importances_, 
    index=dt_model.feature_names_in_
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
dt_importances.plot.barh(ax=ax, color='#3498db')
ax.set_title('Decision Tree Feature Importance')
plt.tight_layout()
plt.savefig("outputs/images/decision_tree_importances.jpg", dpi=150)
plt.show()


### 7.3 Random Forest Feature Importance


In [ ]:
rf_importances = pd.Series(
    rf_model.feature_importances_, 
    index=rf_model.feature_names_in_
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
rf_importances.plot.barh(ax=ax, color='#2ecc71')
ax.set_title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig("outputs/images/random_forest_importances.jpg", dpi=150)
plt.show()


### 7.4 Local Explanation


In [ ]:
chosen_idx = 68
pd.DataFrame(heart_data.iloc[chosen_idx]).T


In [ ]:
instance = preprocessor.transform(X).iloc[chosen_idx:chosen_idx+1]

dt_proba = dt_model.predict_proba(instance)[0, 1]
rf_proba = rf_model.predict_proba(instance)[0, 1]

print(f"Decision Tree: P(disease) = {dt_proba:.2%}")
print(f"Random Forest: P(disease) = {rf_proba:.2%}")


In [ ]:
pd.DataFrame(instance).T


### 7.5 What-If Analysis


In [ ]:
# Counterfactual: what if cholesterol was normalized?
instance_mod = instance.copy()
instance_mod['Cholesterol'] = 5.0

dt_proba_mod = dt_model.predict_proba(instance_mod)[0, 1]
rf_proba_mod = rf_model.predict_proba(instance_mod)[0, 1]

print(f"With Cholesterol=5.0 (normalized):")
print(f"Decision Tree: P(disease) = {dt_proba_mod:.2%}")
print(f"Random Forest: P(disease) = {rf_proba_mod:.2%}")


In [ ]:
pd.DataFrame(instance_mod).T


## 8. Export


In [ ]:
X_train_preprocessed.to_csv("outputs/data/train.csv", index=False)
X_test_preprocessed.to_csv("outputs/data/test.csv", index=False)


---

## Summary

- Random Forest outperforms Decision Tree on accuracy
- Both models significantly beat baseline
- ST_Slope and ChestPainType are top predictive features
- Cost analysis favors models with lower false negative rates
